In [1]:
# **Step 1: 环境准备与依赖导入**
import argparse
import os
import torch
import torch.backends
import random
import numpy as np
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
from exp.exp_imputation import Exp_Imputation
from exp.exp_short_term_forecasting import Exp_Short_Term_Forecast
from exp.exp_anomaly_detection import Exp_Anomaly_Detection
from exp.exp_classification import Exp_Classification
from utils.print_args import print_args

# 固定随机种子
fix_seed = 2021
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

print("环境初始化完成！")

/root/projects/Time-Series-Library/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


环境初始化完成！


In [ ]:
# **Step 2: 参数定义**

class Args:
    # 基本配置
    task_name = 'long_term_forecast'  # 任务名称：long_term_forecast, short_term_forecast, imputation, classification, anomaly_detection
    is_training = 1  # 设置为1则进行训练，设置为0则进行测试
    model_id = 'test'  # 模型编号
    model = 'TimeXer'  # 模型名称：Autoformer, Transformer, TimesNet

    # 数据加载相关
    data = 'custom'  # 数据集类型,选项：[ETTh1,ETTh2,ETTm1,ETTm2,custom,m4,PSM,MSL,SMAP,SMD,SWAT,UEA]
    root_path = './dataset/stock/'  # 数据集根目录路径
    data_path = 'processed_data.csv'  # 数据文件路径
    features = 'MS'  # 预测任务 M:多变量预测多变量, S:单变量预测单变量, MS:多变量预测单变量
    target = 'OT'  # 目标列
    freq = 'd'  # 时间采集粒度，选项：[s:秒, t:分钟, h:小时, d:天, b:工作日, w:周, m:月]
    checkpoints = './checkpoints/'  # 模型存储路径

    # 数据参数
    seq_len = 96 # 输入序列长度,这是用于模型训练的输入序列的长度
    label_len = 48 # 开始标记长度,这是模型输出目标中有标签数据的长度，类似于滑动窗口的长度
    pred_len = 96 # 预测序列长度
    seasonal_patterns = 'Monthly' # 季节模式（针对M4数据集）
    inverse = False # 还原预测数据为原始数据

    # 模型参数
    expand = 2
    d_conv = 4
    top_k = 5 # TimesBlock 中傅里叶变换,频率排名前k个周期
    num_kernels = 6 # Inception 中卷积核个数，用于 Inception
    enc_in = 27 # encoder 输入特征数
    dec_in = 27 # decoder 输入特征数
    c_out = 1 # 输出通道数（MS: 多变量预测单变量，输出为1）
    d_model = 512 # 模型维度，模型中所有子层（包括自注意力层）的输出维度，设置在256至512之间，数字越大显存用的越多
    n_heads = 8 # 多头注意力机制
    e_layers = 2 # encoder 层数，编码器层数
    d_layers = 1 # decoder 层数，解码器层数
    d_ff = 2048 # 前馈网络（FFN）的维度，每个注意力层之后提供额外的表达能力，d_ff是d_model的两到四倍。
    moving_avg = 25 # 滑动窗口长度
    factor = 1 # 对 Q 进行采样，注意力因子数量
    distil = True # 是否下采样操作 pooling，也就是对模型进行压缩
    dropout = 0.1 # dropout 率
    embed = 'timeF' # 时间特征嵌入方式,选项：[timeF, fixed, learned]
    activation = 'gelu' # 激活函数类型
    channel_independence = 1
    decomp_method = 'moving_avg'
    use_norm = 1
    down_sampling_layers = 0
    down_sampling_window = 1
    down_sampling_method = None
    seg_len = 96

    # 优化参数
    num_workers = 10 # cpu并行核心数
    itr = 1 # 实验轮数
    train_epochs = 10 # 训练迭代次数
    batch_size = 32 # batch size 大小
    patience = 3 # early stopping 机制容忍次数
    learning_rate = 0.0001
    des = 'test' # 模型描述
    loss = 'MSE' # 损失函数
    lradj = 'type1' # 学习率下降策略
    use_amp = False

    # GPU 配置
    use_gpu = True
    gpu = 0
    gpu_type = 'cuda'
    use_multi_gpu = False
    devices = '0' # 0,1,2,3  多显卡设置

    # 额外配置
    p_hidden_dims = [128, 128] # 投影仪的隐藏层维度（列表）
    p_hidden_layers = 2 # 投影仪中的隐藏层数
    use_dtw = False
    augmentation_ratio = 0
    seed = 2
    jitter = False
    scaling = False
    permutation = False
    randompermutation = False
    magwarp = False
    timewarp = False
    windowslice = False
    windowwarp = False
    rotation = False
    spawner = False
    dtwwarp = False
    shapedtwwarp = False
    wdba = False
    discdtw = False
    discsdtw = False
    extra_tag = ""
    patch_len = 16


args = Args()
print("参数定义完成！")

参数定义完成！


In [3]:
# **Step 3: 检测 GPU 与设备配置**
if torch.cuda.is_available() and args.use_gpu:
    args.device = torch.device('cuda:{}'.format(args.gpu))
    print('Using GPU')
else:
    if hasattr(torch.backends, "mps"):
        args.device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    else:
        args.device = torch.device("cpu")
    print('Using CPU or MPS')

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(' ', '')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

print(f"使用设备：{args.device}")

Using GPU
使用设备：cuda:0


In [4]:
# **Step 4: 对应任务类型选择实验类**
if args.task_name == 'long_term_forecast':
    Exp = Exp_Long_Term_Forecast
elif args.task_name == 'short_term_forecast':
    Exp = Exp_Short_Term_Forecast
elif args.task_name == 'imputation':
    Exp = Exp_Imputation
elif args.task_name == 'anomaly_detection':
    Exp = Exp_Anomaly_Detection
elif args.task_name == 'classification':
    Exp = Exp_Classification
else:
    Exp = Exp_Long_Term_Forecast

print(f"实验类型选择为：{args.task_name}")

实验类型选择为：long_term_forecast


In [5]:
# **Step 5: 模型训练与测试**

if args.is_training:
    for ii in range(args.itr):
        # 设置实验
        exp = Exp(args)
        setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_expand{}_dc{}_fc{}_eb{}_dt{}_{}_{}'.format(
            args.task_name, args.model_id, args.model, args.data, args.features, args.seq_len, args.label_len,
            args.pred_len, args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.expand,
            args.d_conv, args.factor, args.embed, args.distil, args.des, ii)
        
        print(f">>>>>>>start training : {setting}>>>>>>>>>>>>>>>>>>>>>>>>>>")
        exp.train(setting)

        print(f">>>>>>>testing : {setting}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")
        exp.test(setting)
        
        if args.gpu_type == 'mps':
            torch.backends.mps.empty_cache()
        elif args.gpu_type == 'cuda':
            torch.cuda.empty_cache()
else:
    exp = Exp(args)
    ii = 0
    setting = '{}_{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_expand{}_dc{}_fc{}_eb{}_dt{}_{}_{}'.format(
        args.task_name, args.model_id, args.model, args.data, args.features, args.seq_len, args.label_len, 
        args.pred_len, args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.expand, 
        args.d_conv, args.factor, args.embed, args.distil, args.des, ii)

    print(f">>>>>>>testing : {setting}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")
    exp.test(setting, test=1)
    
    if args.gpu_type == 'mps':
        torch.backends.mps.empty_cache()
    elif args.gpu_type == 'cuda':
        torch.cuda.empty_cache()

print("实验结束！")

Use GPU: cuda:0


NotImplementedError: Model [TimesXer] not found in 'models' directory.